# Checkpoint 1 – Personal data exploration of three people's Fitbit step data

**Author:** Ritishka Gupta (individual submission, no group). Code drafted with help from GenAI (Claude); see the help statement on my Personal Planning and Progress wiki page.

## Overview
**Driving problem:** *Do they achieve at least 7,000 steps on most days by avoiding long periods of inactivity?*

This notebook explores the step data of three people from the FitBit Fitness Tracker Data (Kaggle, CC0 Public Domain), using only the three supplied files: `dailySteps_merged.csv`, `hourlySteps_merged.csv` and `minuteStepsWide_merged.csv`.

For each of the three people it reports:
- (1) the number of days of data for this person
- (2) daily step count information
- (3) minute step count information

Sections: initial assumptions and predictions → load and check the data → reshape the minute data → functions that compute the report → report for each person → what I learnt and how it relates to the driving problem.

## Initial assumptions and predictions
These were written **before** running the code, based on the dataset description and my manual scrutiny of the CSV files in Excel.

**Assumptions**
- A1. `Id` identifies the same person in all three files.
- A2. Each person has about one month (≈31 days) of data from 2016.
- A3. A daily `StepTotal` of 0 most likely means the Fitbit was **not worn**, not that the person took no steps.
- A4. In `minuteStepsWide_merged.csv` each row is one hour, and `Steps00`–`Steps59` are the 60 minutes of that hour. An hour with no row counts as **missing data**. A 0 in a minute cell could mean sitting still or not wearing the device; the data cannot tell these apart.
- A5. For the driving problem I measure a "long period of inactivity" as the **longest run of consecutive zero-step minutes between 07:00 and 21:59** (waking hours), so that sleep is not counted as inactivity.

**Predictions**
- P1. The three people will differ clearly: at least one will average above 7,000 steps per day and at least one below.
- P2. Most minutes will be zero-step minutes, even for an active person.
- P3. Days that reach 7,000 steps will have shorter long inactive periods in waking hours than days that do not.

## Setup
Imports and constants are in one place, so the three people or the step goal can be changed without editing the analysis code.

**Choice of the three people:** during manual scrutiny I compared each person's average daily steps and number of days. I chose a **high**, a **middle** and a **low** average stepper who all have 31 days of data and no 0-step days, so that the comparison for the driving problem is not distorted by days when the device was not worn.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data")
PERSON_IDS = [8877689391, 4558609924, 2320127002]  # high, middle and low average steppers
STEP_GOAL = 7000              # daily goal from the driving problem
WAKE_START, WAKE_END = 7, 22  # waking hours: 07:00 up to (not including) 22:00

## Load and check the data
**Step:** load the three files and confirm their size and columns.
**Assumption:** the files match what I saw in Excel: `Id` plus a date column, and one step column (daily and hourly) or 60 minute columns (minute file).

In [2]:
daily = pd.read_csv(DATA_DIR / "dailySteps_merged.csv")
hourly = pd.read_csv(DATA_DIR / "hourlySteps_merged.csv")
minute_wide = pd.read_csv(DATA_DIR / "minuteStepsWide_merged.csv")

for name, df in {"daily": daily, "hourly": hourly, "minute (wide)": minute_wide}.items():
    print(f"{name:14s} rows={len(df):>6,}  columns={df.shape[1]:>3}  people={df['Id'].nunique()}")

daily          rows=   940  columns=  3  people=33
hourly         rows=22,099  columns=  3  people=33
minute (wide)  rows=21,645  columns= 62  people=33


**Conclusion:** all three files loaded as expected. There are **33 people** in each file, not the 30 given in the dataset description. The minute file has 62 columns (`Id`, `ActivityHour` and 60 minute columns), confirming assumption A4.

**Step:** convert the text dates to real dates, then check for missing values, duplicate rows and negative step counts.
**Assumption:** the dates are in US format (month/day/year), as seen in Excel.

In [3]:
daily["ActivityDay"] = pd.to_datetime(daily["ActivityDay"], format="%m/%d/%Y")
hourly["ActivityHour"] = pd.to_datetime(hourly["ActivityHour"], format="%m/%d/%Y %I:%M:%S %p")
minute_wide["ActivityHour"] = pd.to_datetime(minute_wide["ActivityHour"], format="%m/%d/%Y %I:%M:%S %p")

step_cols = [c for c in minute_wide.columns if c.startswith("Steps")]
pd.DataFrame({
    "empty cells": [daily.isna().sum().sum(), hourly.isna().sum().sum(), minute_wide.isna().sum().sum()],
    "duplicate rows": [daily.duplicated(["Id", "ActivityDay"]).sum(),
                       hourly.duplicated(["Id", "ActivityHour"]).sum(),
                       minute_wide.duplicated(["Id", "ActivityHour"]).sum()],
    "negative steps": [(daily["StepTotal"] < 0).sum(), (hourly["StepTotal"] < 0).sum(),
                       (minute_wide[step_cols] < 0).sum().sum()],
    "first date": [daily["ActivityDay"].min(), hourly["ActivityHour"].min(), minute_wide["ActivityHour"].min()],
    "last date": [daily["ActivityDay"].max(), hourly["ActivityHour"].max(), minute_wide["ActivityHour"].max()],
}, index=["daily", "hourly", "minute (wide)"])

,empty cells,duplicate rows,negative steps,first date,last date
daily,0,0,0,2016-04-12,2016-05-12 00:00:00
hourly,0,0,0,2016-04-12,2016-05-12 15:00:00
minute (wide),0,0,0,2016-04-13,2016-05-13 08:00:00


**Conclusion:** there are no empty cells, duplicate rows or negative step counts, so no values need to be removed. However, the files **cover different dates**. The daily file runs from 12 Apr to 12 May 2016, the hourly file stops at 12 May 15:00, and the minute file starts a day later (13 Apr) and continues to 13 May 08:00. This means 12 Apr has no minute data, and 12 May is probably a partial day in the daily file.

## Reshape the minute data for the three people
**Step:** keep only the three people, then turn the wide minute file (one row per hour, 60 columns) into a long table with one row per minute and a full timestamp. This makes it easy to count minutes and find runs of inactivity.
**Assumption:** the last two characters of each column name give the minute (`Steps07` is minute 7 of that hour).

In [4]:
daily3 = daily[daily["Id"].isin(PERSON_IDS)]

minute3 = minute_wide[minute_wide["Id"].isin(PERSON_IDS)].melt(
    id_vars=["Id", "ActivityHour"], var_name="col", value_name="Steps")
minute3["Timestamp"] = minute3["ActivityHour"] + pd.to_timedelta(minute3["col"].str[-2:].astype(int), unit="min")
minute3 = minute3.drop(columns="col").sort_values(["Id", "Timestamp"]).reset_index(drop=True)

print(f"minute rows for the three people: {len(minute3):,}")
minute3.head(3)

minute rows for the three people: 130,440


,Id,ActivityHour,Steps,Timestamp
0,2320127002,2016-04-13,0,2016-04-13 00:00:00
1,2320127002,2016-04-13,0,2016-04-13 00:01:00
2,2320127002,2016-04-13,0,2016-04-13 00:02:00


**Conclusion:** the reshape gives 130,440 minute rows for the three people. That is close to the predicted ≈ 3 × 31 × 1,440, and the difference comes from the missing 12 Apr and the partial last day. Each row now has a real timestamp.

**Step:** check that the files agree by comparing each day's `StepTotal` in the daily file with the sum of that day's minutes.
**Assumption:** the daily file is the sum of the minute data, so the two totals should be equal.

In [5]:
minute_day_totals = (minute3.groupby(["Id", minute3["Timestamp"].dt.normalize()])["Steps"].sum()
                     .rename("minute_total").reset_index().rename(columns={"Timestamp": "ActivityDay"}))
compare = daily3.merge(minute_day_totals, on=["Id", "ActivityDay"], how="outer")
compare["difference"] = compare["StepTotal"] - compare["minute_total"]
print("days where daily total ≠ sum of minutes:",
      compare[compare["difference"].ne(0)].groupby("Id").size().to_dict())
compare[compare["difference"].ne(0)].sort_values(["Id", "ActivityDay"])

days where daily total ≠ sum of minutes: {2320127002: 9, 4558609924: 5, 8877689391: 28}


,Id,ActivityDay,StepTotal,minute_total,difference
0,2320127002,2016-04-12,10725.0,NaN,NaN
4,2320127002,2016-04-16,5057.0,5044.0,13.0
6,2320127002,2016-04-18,6559.0,6538.0,21.0
8,2320127002,2016-04-20,7192.0,7162.0,30.0
12,2320127002,2016-04-24,4165.0,4159.0,6.0
18,2320127002,2016-04-30,4571.0,4558.0,13.0
26,2320127002,2016-05-08,5161.0,5153.0,8.0
30,2320127002,2016-05-12,2661.0,5543.0,-2882.0
31,2320127002,2016-05-13,NaN,0.0,NaN
32,4558609924,2016-04-12,5135.0,NaN,NaN


**Conclusion:** my assumption was only partly right. On most days the totals match, but the daily file is often a few steps higher than the sum of minutes (by 6–122 steps), so the two files are calculated slightly differently. There are two larger problems. 12 Apr has a daily total but no minute data, and on 12 May the daily total is *lower* than the minute sum (e.g. 6,307 vs 9,995 for Id 4558609924), which confirms that 12 May is a partial day in the daily file. I therefore use the daily file for daily totals but treat 12 Apr and 12 May with caution.

## Functions that compute the report
To avoid repeating code for each person, all statistics come from these functions:
- `longest_zero_run(steps)`: length of the longest block of consecutive zero-step minutes.
- `waking_idle_runs(pid)`: for each day, the longest zero-step run in waking hours (assumption A5).
- `daily_info(pid)`: (1) the number of days and (2) daily step count information.
- `minute_info(pid)`: (3) minute step count information. **Missing data** is counted as days in the daily file with no minute data, plus any empty minute cells.
- `report_person(pid)`: prints the report using the specification's wording.

In [6]:
def longest_zero_run(steps):
    # Give each run of zero / non-zero minutes its own label, then find the longest zero run
    is_zero = steps.eq(0)
    run_label = is_zero.ne(is_zero.shift()).cumsum()
    return int(is_zero.groupby(run_label).sum().max())


def waking_idle_runs(pid):
    # Longest zero-step run between WAKE_START and WAKE_END for each day of this person
    m = minute3[(minute3["Id"] == pid) & minute3["Timestamp"].dt.hour.between(WAKE_START, WAKE_END - 1)]
    return m.groupby(m["Timestamp"].dt.normalize())["Steps"].apply(longest_zero_run)


def daily_info(pid):
    d = daily3[daily3["Id"] == pid]
    return {
        "days": d["ActivityDay"].nunique(),
        "mean": d["StepTotal"].mean(),
        "max": d["StepTotal"].max(),
        "max_day": d.loc[d["StepTotal"].idxmax(), "ActivityDay"],
        "min": d["StepTotal"].min(),
        "min_day": d.loc[d["StepTotal"].idxmin(), "ActivityDay"],
        "goal_days": int((d["StepTotal"] >= STEP_GOAL).sum()),
        "zero_days": int((d["StepTotal"] == 0).sum()),
    }


def minute_info(pid):
    m = minute3[minute3["Id"] == pid]
    first, last = m["ActivityHour"].min(), m["ActivityHour"].max()
    daily_days = set(daily3.loc[daily3["Id"] == pid, "ActivityDay"])
    days_without_minutes = sorted(daily_days - set(m["Timestamp"].dt.normalize()))
    idle_runs = waking_idle_runs(pid)
    return {
        "minutes": len(m),
        "nonzero": int((m["Steps"] > 0).sum()),
        "missing_days": days_without_minutes,
        "empty_cells": int(m["Steps"].isna().sum()),
        "first": first,
        "last": last,
        "mean": m["Steps"].mean(),
        "mean_nonzero": m.loc[m["Steps"] > 0, "Steps"].mean(),
        "max": int(m["Steps"].max()),
        "max_time": m.loc[m["Steps"].idxmax(), "Timestamp"],
        "min": int(m["Steps"].min()),
        "idle_median": idle_runs.median(),
        "idle_max": idle_runs.max(),
    }

In [7]:
def report_person(pid):
    d, m = daily_info(pid), minute_info(pid)
    print(f"Person Id {pid}\n")
    print(f"(1) the number of days of data for this person: {d['days']}\n")
    print("(2) daily step count information")
    print(f"  average step count per day: {d['mean']:,.0f} steps")
    print(f"  maximum step count: {d['max']:,} steps ({d['max_day']:%a %d %b %Y})")
    print(f"  minimum step count: {d['min']:,} steps ({d['min_day']:%a %d %b %Y})")
    print(f"  one other observation you made about the data for this person: "
          f"{d['goal_days']} of {d['days']} days ({d['goal_days'] / d['days']:.0%}) reached "
          f"{STEP_GOAL:,} steps; {d['zero_days']} day(s) had 0 steps (likely not worn)\n")
    print("(3) minute step count information")
    print(f"  number of non-zero minutes: {m['nonzero']:,} of {m['minutes']:,} recorded minutes "
          f"({m['nonzero'] / m['minutes']:.0%})")
    missing = ", ".join(f"{day:%d %b}" for day in m["missing_days"]) or "none"
    print(f"  missing data: {len(m['missing_days'])} day(s) of the daily file have no minute data "
          f"({missing}), i.e. {len(m['missing_days']) * 1440:,} minutes; {m['empty_cells']} empty "
          f"minute cell(s); minute data runs {m['first']:%d %b %H:%M} to {m['last']:%d %b %H:%M}")
    print(f"  average steps per minute: {m['mean']:.1f} over all recorded minutes "
          f"({m['mean_nonzero']:.1f} over non-zero minutes only)")
    print(f"  maximum and minimum steps: maximum {m['max']} steps in one minute "
          f"({m['max_time']:%a %d %b %H:%M}), minimum {m['min']}")
    print(f"  one other observation you made about the data for this person: longest run of "
          f"zero-step minutes in waking hours is {m['idle_median']:.0f} min on a typical day "
          f"(median), and {m['idle_max']:.0f} min at most")

## Person 1: Id 8877689391
**Prediction:** the most active of the three (highest average in my manual scrutiny), so I expect 7,000 steps on almost every day, a high maximum steps per minute (running) and short inactive periods during the day.

In [8]:
report_person(PERSON_IDS[0])

Person Id 8877689391

(1) the number of days of data for this person: 31

(2) daily step count information
  average step count per day: 16,040 steps
  maximum step count: 29,326 steps (Sat 16 Apr 2016)
  minimum step count: 4,790 steps (Mon 02 May 2016)
  one other observation you made about the data for this person: 30 of 31 days (97%) reached 7,000 steps; 0 day(s) had 0 steps (likely not worn)

(3) minute step count information
  number of non-zero minutes: 9,230 of 43,680 recorded minutes (21%)
  missing data: 1 day(s) of the daily file have no minute data (12 Apr), i.e. 1,440 minutes; 0 empty minute cell(s); minute data runs 13 Apr 00:00 to 13 May 07:00
  average steps per minute: 11.0 over all recorded minutes (52.3 over non-zero minutes only)
  maximum and minimum steps: maximum 187 steps in one minute (Wed 20 Apr 18:35), minimum 0
  one other observation you made about the data for this person: longest run of zero-step minutes in waking hours is 48 min on a typical day (median)

**Interpretation:** Prediction confirmed. This person reached 7,000 steps on 30 of 31 days (97%). Their only miss, Mon 02 May (4,790 steps), was also the day with their longest waking inactive period (446 min), which fits the driving problem. At the minute level, only 21% of minutes had any steps, but when moving they averaged 52 steps per minute, about twice the other two people. On a typical day their longest inactive period in waking hours was under an hour (48 min). The daily file was slightly higher than the sum of minutes on 28 days (by up to 122 steps), so the daily and minute files are not calculated in exactly the same way.

## Person 2: Id 4558609924
**Prediction:** a borderline person whose average is close to 7,000, so I expect about half of the days to reach the goal and longer inactive periods than Person 1.

In [9]:
report_person(PERSON_IDS[1])

Person Id 4558609924

(1) the number of days of data for this person: 31

(2) daily step count information
  average step count per day: 7,685 steps
  maximum step count: 13,743 steps (Thu 21 Apr 2016)
  minimum step count: 3,428 steps (Sun 01 May 2016)
  one other observation you made about the data for this person: 18 of 31 days (58%) reached 7,000 steps; 0 day(s) had 0 steps (likely not worn)

(3) minute step count information
  number of non-zero minutes: 9,193 of 43,020 recorded minutes (21%)
  missing data: 1 day(s) of the daily file have no minute data (12 Apr), i.e. 1,440 minutes; 0 empty minute cell(s); minute data runs 13 Apr 00:00 to 12 May 20:00
  average steps per minute: 5.5 over all recorded minutes (25.8 over non-zero minutes only)
  maximum and minimum steps: maximum 207 steps in one minute (Wed 27 Apr 18:46), minimum 0
  one other observation you made about the data for this person: longest run of zero-step minutes in waking hours is 134 min on a typical day (median),

**Interpretation:** Prediction confirmed. 18 of 31 days (58%) reached 7,000 steps, so this person meets the goal on most days, but only just. Their last day (12 May, 6,307 steps) is a partial day in the daily file: the minute data for that day sums to 9,995 steps, so the true count is probably 19 of 31. They had non-zero steps in 21% of minutes, the same share as Person 1, but moved more slowly (26 steps per non-zero minute). Their typical longest inactive period in waking hours was 134 min, almost three times Person 1's.

## Person 3: Id 2320127002
**Prediction:** the least active of the three, so I expect few days at 7,000 steps, a low share of non-zero minutes and long inactive periods during the day.

In [10]:
report_person(PERSON_IDS[2])

Person Id 2320127002

(1) the number of days of data for this person: 31

(2) daily step count information
  average step count per day: 4,717 steps
  maximum step count: 10,725 steps (Tue 12 Apr 2016)
  minimum step count: 772 steps (Sun 01 May 2016)
  one other observation you made about the data for this person: 5 of 31 days (16%) reached 7,000 steps; 0 day(s) had 0 steps (likely not worn)

(3) minute step count information
  number of non-zero minutes: 6,079 of 43,740 recorded minutes (14%)
  missing data: 1 day(s) of the daily file have no minute data (12 Apr), i.e. 1,440 minutes; 0 empty minute cell(s); minute data runs 13 Apr 00:00 to 13 May 08:00
  average steps per minute: 3.2 over all recorded minutes (22.7 over non-zero minutes only)
  maximum and minimum steps: maximum 123 steps in one minute (Tue 26 Apr 22:17), minimum 0
  one other observation you made about the data for this person: longest run of zero-step minutes in waking hours is 302 min on a typical day (median), an

**Interpretation:** Prediction confirmed. Only 5 of 31 days (16%) reached 7,000 steps, and only 14% of minutes had any steps. Their highest day (12 Apr, 10,725 steps) cannot be checked because the minute file has no data for 12 Apr. Their lowest day, Sun 01 May (772 steps), had **zero steps in every minute from 07:00 to 21:59** (a 900-minute run), which suggests the device was not worn for most of the day. So non-wear is not limited to 0-step days, which weakens assumption A3. Their typical longest inactive period was about 5 hours (302 min). Their busiest hours were 20:00–21:00, so the fixed 07:00–21:59 waking window may not suit this person.

## Link to the driving problem: long inactive periods on goal days and other days
**Step:** for each person, compare the longest waking-hours zero-step run on days that reached 7,000 steps with days that did not, and measure the correlation between the two.
**Assumption:** only full days are used. 12 Apr (no minute data) and 12 May (a partial day in the daily file) are excluded.

In [11]:
full_days = daily3[(daily3["ActivityDay"] > "2016-04-12") & (daily3["ActivityDay"] < "2016-05-12")]
rows = []
for pid in PERSON_IDS:
    days = full_days[full_days["Id"] == pid].set_index("ActivityDay")
    days["idle"] = waking_idle_runs(pid)
    goal = days["StepTotal"] >= STEP_GOAL
    rows.append({
        "Id": pid,
        "goal days / full days": f"{goal.sum()} / {len(days)}",
        "median longest idle run, goal days (min)": days.loc[goal, "idle"].median(),
        "median longest idle run, other days (min)": days.loc[~goal, "idle"].median(),
        "correlation: idle run vs daily steps": round(days["idle"].corr(days["StepTotal"]), 2),
    })
pd.DataFrame(rows).set_index("Id")

,goal days / full days,"median longest idle run, goal days (min)","median longest idle run, other days (min)",correlation: idle run vs daily steps
Id,,,,
8877689391,28 / 29,48.0,446.0,-0.37
4558609924,18 / 29,74.5,189.0,-0.63
2320127002,4 / 29,244.5,413.0,-0.80


**Conclusion:** prediction P3 is supported for all three people. On days that reached 7,000 steps, the longest waking inactive period was much shorter (median 48 vs 446 min, 75 vs 189 min, and 245 vs 413 min). The correlation between the longest inactive period and daily steps is negative for everyone (−0.37, −0.63 and −0.80). Person 1 has only one non-goal day, so their comparison rests on a single day. This is an association in three people, not proof that avoiding inactivity *causes* more steps.

## What I learnt from this data exploration and how it relates to the driving problem
**What I learnt about the data**
- The files cover **33 people**, have no empty, duplicate or negative values, and do **not** line up exactly. The minute file has no 12 Apr data, the daily file's 12 May is a partial day, and daily totals are often slightly higher than the sum of minutes.
- **A 0-step day is not the only kind of non-wear.** Id 2320127002 had 772 steps on 01 May but zero steps in every waking minute, so they probably wore the device for only a small part of the day. Assumption A3 was too simple.
- **Most minutes are zero-step minutes for everyone** (79–86%), which confirms P2. The three people differ less in *how often* they move (21%, 21% and 14% of minutes) than in *how fast* they move (52, 26 and 23 steps per non-zero minute) and in *how long* their inactive periods last.

**How this relates to the driving problem:** *Do they achieve at least 7,000 steps on most days by avoiding long periods of inactivity?*
- **7,000 steps on most days:** taking "most days" as more than half, two of the three people meet this (97% and 58% of days) and one does not (16%). This confirms P1.
- **Avoiding long periods of inactivity:** for all three people, days with a shorter longest inactive period in waking hours had more steps, and goal days had much shorter inactive periods than other days. This suggests the answer for these three is "yes, the people and days that reach 7,000 steps are those with shorter inactive periods". The link is an association, not proof of cause.

**What this means for the next stage of the analysis**
1. Exclude or flag 12 Apr (no minute data) and 12 May (partial day) when counting goal days.
2. Define a non-wear rule that also catches partially worn days (for example, a whole waking window of zero steps), not just 0-step days.
3. Agree clear definitions of "most days" and of a "long period of inactivity" (for example, a zero-step run of at least 60 minutes), with a source for each.
4. A fixed 07:00–21:59 waking window does not suit everyone (Id 2320127002 is most active at 20:00–21:00), and these files contain no sleep data to set it per person. This is a limitation to state.
5. Extend the functions from three people to every person in the dataset.